# Modelling Quantum Fluids with the Gross-Pitaevskii Equation

This notebook runs a 2D **Gross-Pitaevskii Equation (GPE)** simulator (`GrossPitaevskiiEquation2D`). This equation models the dynamics of Bose-Einstein Condensates (BECs) and superfluid helium (quantum fluids).

- **State variables**: `density` ($|\psi|^2$), `real` ($\Re(\psi)$), and `imag` ($\Im(\psi)$).
- **Conditioning variables**: `g` controls non-linear interactions, while `disorder_strength`, `spoon_strength`, `wx`, and `wy` adjust the trap potential geometry and complexities.
- **Dynamics**: Split-Step Fourier Method alternating between real (potential and interaction) and momentum (kinetic) space.
- **Boundary conditions**: Naturally periodic via the Fast Fourier Transform (FFT) combined with localized harmonic trapping.

### Why this is useful

In classical fluid dynamics datasets, models learn to predict pressure and velocity. In quantum mechanics:

- **Density** acts analogously to "pressure/mass".
- **Real/imag** provide a smooth representation of the wavefunction and avoid branch-cut discontinuities from wrapped phase.
- **Phase** is still recoverable as `atan2(imag, real)` when needed.

By manipulating the complexity knobs in the potential trap (like adding a "laser spoon" to stir the fluid or "optical speckle" for spatial disorder), we can generate anywhere from simple harmonic oscillations to chaotic quantum turbulence.


## Governing equations

The Gross-Pitaevskii Equation describes the ground state of a quantum system of identical bosons using a pseudo-wavefunction $\psi(x,y,t)$.

$$
i \hbar \frac{\partial \psi}{\partial t} = \left( -\frac{\hbar^2}{2m}\nabla^2 + V(x,y,t) + g |\psi|^2 - \Omega L_z \right) \psi
$$

Where:

- $\nabla^2$ is the kinetic energy operator.
- $V(x,y,t)$ is the external trapping potential (harmonic trap, obstacle, and/or disorder).
- $g|\psi|^2$ represents the non-linear inter-particle interaction (e.g. atomic repulsion).
- $\Omega L_z$ is the rotating-frame energy term, where $\Omega$ is the frame angular velocity and $L_z = -i\hbar(x\partial_y - y\partial_x)$ is the $z$-component of angular momentum. Setting $\Omega > 0$ nucleates quantised vortices in the condensate.

In this simulator, we use dimensionless units where $\hbar = 1$ and $m = 1$. The wave function is stepped forward in time using the **Split-Step Fourier Method** due to its unconditional numerical stability and probability conservation compared to standard ODE solvers.


### Simulation Parameters Explained

In both the low and high complexity examples, we configure parameters that control physically meaningful regimes and dataset diversity:

- **`g`**: Non-linear interaction strength. $g>0$ models repulsive bosons; $g=0$ recovers the linear Schrödinger limit.
- **`disorder_strength`**: Amplitude of a spatially random, speckle-like potential (kept stationary by default for physical consistency).
- **`spoon_strength`**: Amplitude of a localized Gaussian obstacle ("laser spoon").
- **`spoon_speed`**: Angular speed of the spoon trajectory in the lab-frame potential.
- **`Omega`**: Rotating-frame angular velocity term ($-\Omega L_z$), independent of spoon motion.
- **`wx`, `wy`**: Harmonic trap frequencies; larger values produce tighter confinement.
- **`x0`, `y0`**: Initial condensate center.
- **`kx0`, `ky0`**: Initial momentum kick.
- **`width`**: Initial Gaussian width (smaller values induce stronger breathing).
- **`imaginary_time`**: If enabled, evolves in imaginary time to relax toward low-energy states (with renormalization each step).

### Physical modeling scope

This simulator is a mean-field $T\approx 0$ model (Gross-Pitaevskii Equation). External laser/disorder effects are represented as classical potentials in $V(x,y,t)$. It does **not** explicitly model decoherence, thermal clouds, or stochastic dissipation; those require extensions such as SGPE/ZNG-type models.

In [ ]:
from IPython.display import HTML

from autosim.experimental.simulations import GrossPitaevskiiEquation2D as GPESim
from autosim.utils import plot_spatiotemporal_video


### Low Complexity Example

When $g=0$, the system reduces to the linear Schrödinger equation. With a perfectly round harmonic trap (`wx=1.0, wy=1.0`), the wave behaves highly predictably.

If we place a Gaussian wavepacket inside this trap, its behavior depends heavily on its initial `width`:

- **Coherent state (`width=1.0`)**: If the initial width perfectly matches the trap's natural length scale ($1/\sqrt{w_x} = 1.0$), it acts exactly like a classical ball on a spring. It will slosh back and forth forever without its shape changing or spreading out.
- **Squeezed state (`width=0.4`)**: By squishing the initial wavepacket so its `width` is thinner than the trap's natural scale, we force it to disperse (spread out) as it moves. Because it is trapped, it bounces off the edges and folds back in on itself.

**What you will see:**

1. **Sloshing:** The center of the wavepacket swings side-to-side in the trap.
2. **Breathing:** The wavepacket alternately expands (spreads out) and contracts (squeezes back together) as it moves.
3. **Interference fringes:** As the spreading wavepacket hits the trap walls and reflects back over itself, the overlapping waves create high-contrast ripples (constructive and destructive interference).


In [ ]:
sim_easy = GPESim(
    return_timeseries=True,
    log_level="progress_bar",
    n=128,  # grid resolution 128x128
    T=24.0,  # 24/3.14 ≈ 7.64 breathing periods, ~4 full periods (T_trap = 2π ≈ 6.28)
    dt=0.005,
    snapshot_dt=0.1,  # finer snapshots to capture the breathing clearly
    L=10.0,
    parameters_range={
        "g": (0.0, 0.0),
        "disorder_strength": (0.0, 0.0),
        "spoon_strength": (0.0, 0.0),
        "wx": (1.0, 1.0),
        "wy": (1.0, 1.0),
        # Matplotlib imshow displays the mathematical Y-axis vertically by default.
        # Simulation creates grids using ij indexing, meaning X is row (vertical).
        # Push the packet along the y-axis making it slosh purely horizontally visually.
        "x0": (0.0, 0.0),
        "y0": (1.5, 1.5),
        "kx0": (0.0, 0.0),
        "ky0": (-2.0, -2.0),
        # width < a_ho = 1/sqrt(wx) = 1.0 => squeezed state: packet breathes and
        # spreads, producing visible interference fringes as it overlaps with itself
        "width": (0.4, 0.4),
    },
    random_seed=42,
)

res_easy = sim_easy.forward_samples_spatiotemporal(n=1)

print("Constant Scalars (Batch x Params):", res_easy["constant_scalars"].shape)
print(
    "Spatiotemporal Output:",
    res_easy["data"].shape,
    "[batch, time, x, y, channels={density, real, imag}]",
)


In [ ]:
anim_easy = plot_spatiotemporal_video(
    res_easy["data"],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
)
HTML(anim_easy.to_jshtml())

### High Complexity / Turbulence Example

The simulation parameters can be tuned to produce strongly non-linear, vortex-rich dynamics useful for ML forecasting benchmarks.

- **Strong Repulsion (`g=50`)**: Increases nonlinear pressure and suppresses simple overlap dynamics.
- **Disorder (`disorder_strength=0.8`)**: Static speckle-like landscape that scatters and fragments flow.
- **Laser Spoon (`spoon_strength=3.0`, `spoon_speed=1.5`)**: A moving obstacle that injects energy and sheds wakes/vortices.
- **Rotating Frame (`Omega=0.2`)**: Adds rotational bias through the $-\Omega L_z$ term.
- **Tighter Trap (`wx=2.0`, `wy=2.0`)**: Raises central density and strengthens nonlinear interactions.

**What you will see:**

1. **Wake formation and shear:** Flow wraps around the moving obstacle and forms complex trailing structures.
2. **Density fragmentation:** Speckle scattering and stirring produce intermittent low/high-density patches.
3. **Wavefunction winding:** Vortex cores appear where density dips and the reconstructed phase (`atan2(imag, real)`) winds by $2\pi$ around the core.

In [ ]:
# Aggressive 64x64 regime for many microvortices.
# Strong stirring + stronger disorder + broader condensate support.
Omega = 1.25
spoon_strength = 14.0
spoon_speed = 10.0
spoon_width = 0.10
spoon_radius = 1.8
trap_strength = 0.9
repulsion_strength = 220.0

sim_hard = GPESim(
    return_timeseries=True,
    log_level="progress_bar",
    n=64,
    T=32.0,
    dt=0.0015,
    snapshot_dt=0.03,
    L=10.0,
    parameters_range={
        "g": (repulsion_strength, repulsion_strength),
        "disorder_strength": (0.9, 0.9),
        "disorder_radius": (0.9, 0.9),
        "disorder_time_dependent": (1, 1),
        "spoon_strength": (spoon_strength, spoon_strength),
        "spoon_speed": (spoon_speed, spoon_speed),
        "spoon_width": (spoon_width, spoon_width),
        "spoon_radius": (spoon_radius, spoon_radius),
        "wx": (trap_strength, trap_strength),
        "wy": (trap_strength, trap_strength),
        "x0": (0.0, 0.0),
        "y0": (0.0, 0.0),
        "kx0": (1.0, 1.0),
        "ky0": (0.0, 0.0),
        "width": (1.3, 1.3),
        "Omega": (Omega, Omega),
        "imaginary_time": (0, 0),
        # "box_param": (5.0, 5.0),
        # "box_param": (0.1, 0.1),
        # "box_param": (0.02, 0.02),
        # "box_param": (0.05, 0.05),
        # "box_param": (0.08, 0.08),
        # More like donut
        # "box_param": (0.1, 0.1),
        # ???
        "box_param": (4.0, 4.0),
    },
    random_seed=42,
)

res_hard = sim_hard.forward_samples_spatiotemporal(n=1)

In [ ]:
res_hard["data"].shape

In [ ]:

anim_hard = plot_spatiotemporal_video(
    res_hard["data"][:, 500:],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
)
HTML(anim_hard.to_jshtml())

In [ ]:
# Stability diagnostics + find frame with maximum detected vortex cores
import matplotlib.pyplot as plt
import torch

density = res_hard["data"][0, ..., 0]
real = res_hard["data"][0, ..., 1]
imag = res_hard["data"][0, ..., 2]

print("density min/max:", float(density.min()), float(density.max()))
print(
    "density p95/p99:",
    float(torch.quantile(density, 0.95)),
    float(torch.quantile(density, 0.99)),
)
print("density max first/last:", float(density[0].max()), float(density[-1].max()))
print("nan/inf:", int(torch.isnan(density).sum()), int(torch.isinf(density).sum()))

def wrap_to_pi(x: torch.Tensor) -> torch.Tensor:
    return (x + torch.pi) % (2 * torch.pi) - torch.pi

def detect_vortex_cores(r: torch.Tensor, i: torch.Tensor, d: torch.Tensor):
    phi = torch.atan2(i, r)
    d1 = wrap_to_pi(phi[:-1, 1:] - phi[:-1, :-1])
    d2 = wrap_to_pi(phi[1:, 1:] - phi[:-1, 1:])
    d3 = wrap_to_pi(phi[1:, :-1] - phi[1:, 1:])
    d4 = wrap_to_pi(phi[:-1, :-1] - phi[1:, :-1])
    winding = d1 + d2 + d3 + d4

    mask_threshold = 0.03 * float(d.max())
    vpos = (winding > torch.pi).nonzero(as_tuple=False)
    vneg = (winding < -torch.pi).nonzero(as_tuple=False)

    if vpos.numel() > 0:
        keep = d[vpos[:, 0], vpos[:, 1]] > mask_threshold
        vpos = vpos[keep]
    if vneg.numel() > 0:
        keep = d[vneg[:, 0], vneg[:, 1]] > mask_threshold
        vneg = vneg[keep]

    return vpos, vneg, mask_threshold

best_t = 0
best_total = -1
best_pos = torch.empty((0, 2), dtype=torch.long)
best_neg = torch.empty((0, 2), dtype=torch.long)
for t in range(density.shape[0]):
    vpos, vneg, _ = detect_vortex_cores(real[t], imag[t], density[t])
    total = int(vpos.shape[0] + vneg.shape[0])
    if total > best_total:
        best_total = total
        best_t = t
        best_pos, best_neg = vpos, vneg

print("best frame index:", best_t, "of", int(density.shape[0]))
print("estimated + vortices:", int(best_pos.shape[0]))
print("estimated - vortices:", int(best_neg.shape[0]))
print("estimated total vortices:", int(best_total))

density_best = density[best_t]
phase_best = torch.atan2(imag[best_t], real[best_t])
mask_threshold = 0.03 * float(density_best.max())
phase_masked = phase_best.clone()
phase_masked[density_best < mask_threshold] = float("nan")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(density_best.cpu(), cmap="magma", origin="lower")
axes[0].set_title(f"Density (frame {best_t})")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(
    phase_masked.cpu(), cmap="twilight", origin="lower", vmin=-torch.pi, vmax=torch.pi
)
axes[1].set_title(f"Phase + detected cores (frame {best_t})")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

if best_pos.numel() > 0:
    axes[1].scatter(
        best_pos[:, 1].cpu(), best_pos[:, 0].cpu(), s=24, c="cyan", marker="+", label="+1"
)
if best_neg.numel() > 0:
    axes[1].scatter(
        best_neg[:, 1].cpu(), best_neg[:, 0].cpu(), s=24, c="yellow", marker="x", label="-1"
)
if best_pos.numel() > 0 or best_neg.numel() > 0:
    axes[1].legend(loc="upper right", fontsize=8)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

### Vortex Lattice Example

A regular Abrikosov-like lattice is a different regime from the wake-shedding `laser_only` family.

Here we remove the spoon and disorder, use a slightly asymmetric elliptical trap (as in the 2001 Ketterle *Science* experiment), and use sustained rotation so vortices can organise rather than being continuously scrambled by obstacle forcing.

In this notebook that rotation comes from a nonzero `Omega`, which represents a rotating-frame / rotating-trap drive rather than a laser stirrer. Physically, this is closer to spinning the confinement than dragging an obstacle through the condensate.

This is best interpreted as a lattice-forming regime rather than a turbulence regime. On a `64 x 64` grid the lattice is quite coarse, so this example uses `128 x 128` to make the ordered cores easier to see.


In [ ]:
sim_lattice = GPESim(
    return_timeseries=True,
    log_level="progress_bar",
    n=128,
    T=24.0,
    dt=0.002,
    snapshot_dt=0.06,
    L=10.0,
    parameters_range={
        "g": (90.0, 90.0),
        "disorder_strength": (0.0, 0.0),
        "disorder_radius": (1.0, 1.0),
        "disorder_time_dependent": (0, 0),
        "spoon_strength": (0.0, 0.0),
        "spoon_speed": (0.0, 0.0),
        "spoon_width": (0.18, 0.18),
        "spoon_radius": (1.5, 1.5),
        "wx": (1.0, 1.0),
        "wy": (1.1, 1.1),
        "x0": (0.0, 0.0),
        "y0": (0.0, 0.0),
        "kx0": (0.0, 0.0),
        "ky0": (0.0, 0.0),
        "width": (1.0, 1.0),
        # "initial_noise": (0.05, 0.05),
        "Omega": (0.75, 0.75),
        "imaginary_time": (0, 0),
        "imaginary_time_steps": (1000, 1000),
        "box_param": (0.2, 0.2),
    },
    random_seed=42,
)

res_lattice = sim_lattice.forward_samples_spatiotemporal(n=1)
print(res_lattice["data"].shape)


In [ ]:
res_lattice["data"].shape

In [ ]:

anim_lattice = plot_spatiotemporal_video(
    res_lattice["data"][:, ::1],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True,
)
HTML(anim_lattice.to_jshtml())


## Dataset Generation

We generate five dataset families on a `64 x 64` grid with `321` returned time points.
Internally we simulate to `T=17.05` with `snapshot_dt=0.05`, which gives `342` saved snapshots including the initial state; the simulator then drops the initial frame plus `skip_nt=20` additional early snapshots, leaving exactly `321` frames for learning.

The aim is to cover a useful spread of Bose-Einstein condensate behaviour rather than only highly vortical states:

1. **`low_complexity`**: near-linear or weakly nonlinear condensates with breathing, sloshing, and interference fringes but little or no vortex activity.
2. **`laser_only`**: obstacle-driven wake formation and controllable vortex shedding, which is the cleanest regime for studying nucleation.
3. **`vortex_lattice`**: near-equilibrium rotating condensates where vortices tend to arrange into coarse ordered lattices instead of disordered wakes.
4. **`speckle_only`**: disorder-driven scattering and fragmentation with weaker, less systematic vortex production.
5. **`high_complexity`**: the richest mixed regime, where obstacle shedding interacts with disorder to create more irregular turbulence.

A practical lesson from the notebook exploration is that the very aggressive setting used for `sim_hard` (`g=220`, `Omega=1.25`, `spoon_strength=14`, `spoon_speed=10`, strong time-dependent disorder) is reasonable as a **vortex stress-test**, but it is too extreme to define the whole experimental range. It tends to collapse the dataset toward one phenomenology: strongly forced, highly nonlinear turbulence.

For a diverse dataset we therefore prefer:

- **Moderate nonlinearity** in the main turbulent families: `g ~= 40-100`.
- **Mild quartic confinement**: `box_param = 0.2`, enough to suppress FFT wrap-around without creating the hollowed-out donut morphology.
- **Laser-only families** for the cleanest vortex exploration.
- **A separate `vortex_lattice` family** for ordered rotating states, with no spoon and no disorder.
- **A `high_complexity` mixed family** as an additional regime, not a replacement, because mixing obstacle forcing and disorder makes interpretation harder even though it broadens phenomenology.
- **A short warm-up skip** (`skip_nt=20`) so the emulator is not forced to predict the earliest symmetry-breaking transient directly from a nearly symmetric initial condition.
- **Speckle-only families** to ensure the dataset also contains non-wake fragmentation and lower-vorticity dynamics.

So the recommended design is to keep both clean and mixed forcing regimes, and to separate wake vortices from lattice vortices rather than expecting one family to do both.

A `quick_demo` switch is used below so the same notebook supports both fast local checks and larger training-scale generation.


In [ ]:
# T=17.05, snapshot_dt=0.05 -> 17.05 / 0.05 = 341 saved intervals.
# Including the initial state gives 342 raw frames; skip_nt=20 then drops the
# initial frame plus 20 more early snapshots, leaving 321 frames per sample.
T_dataset = 0.5
dt = 0.005
snapshot_dt = 0.05
skip_nt = 20
n_grid = 64

quick_demo = True
if quick_demo:
    split_sizes = {"train": 4, "valid": 2, "test": 2}
else:
    split_sizes = {"train": 200, "valid": 20, "test": 20}

# Shared baseline: mild quartic walls keep the condensate blob contained without
# pushing it into the hollow donut morphology seen with very strong forcing.
base_params = {
    "wx": (0.9, 1.2),
    "wy": (0.9, 1.2),
    # "box_param": (0.15, 0.35),
    "box_param": (0.2, 0.2),
    "width": (0.5, 1.1),
    "x0": (-1.0, 1.0),
    "y0": (-1.0, 1.0),
    "kx0": (-2.0, 2.0),
    "ky0": (-2.0, 2.0),
    "imaginary_time": (0, 0),
}

dataset_configs = {
    # Mostly non-vortex dynamics: breathing, sloshing, weak interference, mild nonlinear deformation.
    "low_complexity": {
        **base_params,
        "g": (0.0, 15.0),
        "disorder_strength": (0.0, 0.15),
        "disorder_radius": (0.8, 1.3),
        "disorder_time_dependent": (0, 0),
        "spoon_strength": (0.0, 0.0),
        "spoon_speed": (0.0, 0.0),
        "spoon_width": (0.12, 0.22),
        "spoon_radius": (1.2, 1.9),
        "Omega": (0.0, 0.05),
    },
    # Cleanest family for vortex exploration: obstacle-driven wakes and vortex pair shedding.
    "laser_only": {
        **base_params,
        "g": (50.0, 100.0),
        "disorder_strength": (0.0, 0.0),
        "disorder_radius": (0.8, 1.3),
        "disorder_time_dependent": (0, 0),
        "spoon_strength": (2.5, 6.0),
        "spoon_speed": (1.0, 2.5),
        "spoon_width": (0.12, 0.22),
        "spoon_radius": (1.2, 1.9),
        "Omega": (0.05, 0.35),
    },
    # Ordered rotating condensates: no spoon, no disorder, symmetric trap, moderate
    # rotation.
    "vortex_lattice": {
        **base_params,
        "wx": (1.0, 1.0),
        "wy": (1.05, 1.2),
        "box_param": (0.2, 0.2),
        "width": (0.9, 1.1),
        "x0": (0.0, 0.0),
        "y0": (0.0, 0.0),
        "kx0": (0.0, 0.0),
        "ky0": (0.0, 0.0),
        "g": (70.0, 110.0),
        "disorder_strength": (0.0, 0.0),
        "disorder_radius": (1.0, 1.0),
        "disorder_time_dependent": (0, 0),
        "spoon_strength": (0.0, 0.0),
        "spoon_speed": (0.0, 0.0),
        "spoon_width": (0.16, 0.22),
        "spoon_radius": (1.4, 1.8),
        "Omega": (0.6, 0.8),
        "imaginary_time_steps": (1000, 1000),
    },
    # Disorder-driven fragmentation and scattering with weaker or intermittent
    # vorticity.
    "speckle_only": {
        **base_params,
        "g": (40.0, 90.0),
        "disorder_strength": (0.4, 1.2),
        "disorder_radius": (0.7, 1.4),
        "disorder_time_dependent": (0, 0),
        "spoon_strength": (0.0, 0.0),
        "spoon_speed": (0.0, 0.0),
        "spoon_width": (0.12, 0.22),
        "spoon_radius": (1.2, 1.9),
        "Omega": (0.0, 0.15),
    },
    # Rich mixed regime: keep it, but make it a secondary family rather than the whole
    # dataset identity.
    "high_complexity": {
        **base_params,
        "g": (50.0, 100.0),
        "disorder_strength": (0.4, 1.0),
        "disorder_radius": (0.7, 1.3),
        "disorder_time_dependent": (0, 1),
        "spoon_strength": (2.5, 5.5),
        "spoon_speed": (1.0, 2.2),
        "spoon_width": (0.12, 0.22),
        "spoon_radius": (1.2, 1.9),
        "Omega": (0.1, 0.4),
    },
}

outputs = {}

for name, params in dataset_configs.items():
    print(f"Generating dataset: {name}")
    sim = GPESim(
        return_timeseries=True,
        log_level="warning",
        n=n_grid,
        T=T_dataset,
        dt=dt,
        snapshot_dt=snapshot_dt,
        skip_nt=skip_nt,
        L=10.0,
        parameters_range=params,
        random_seed=42,
    )

    outputs[name] = {}
    for split, count in split_sizes.items():
        print(f"  -> Split: {split} (n={count})")
        res = sim.forward_samples_spatiotemporal(n=count)
        outputs[name][split] = res

print("Finished generating all datasets!")
print(
    "Example shape for low_complexity train:",
    outputs["low_complexity"]["train"]["data"].shape,
)


In [ ]:
from pathlib import Path

import torch

# Save datasets to disk
output_root = Path("./generated_datasets")
for name in dataset_configs:
    for split in ["train", "valid", "test"]:
        save_dir = output_root / f"gpe_{name}" / split
        save_dir.mkdir(parents=True, exist_ok=True)
        output_path = save_dir / "data.pt"
        torch.save(outputs[name][split], output_path)

print(f"Saved all datasets under: {output_root.resolve()}")

In [ ]:
anim = plot_spatiotemporal_video(
    outputs["low_complexity"]["train"]["data"][:, ::4],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True
)
HTML(anim.to_jshtml())

In [ ]:
anim = plot_spatiotemporal_video(
    outputs["laser_only"]["train"]["data"][:, ::2],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
)
HTML(anim.to_jshtml())

In [ ]:
anim = plot_spatiotemporal_video(
    outputs["vortex_lattice"]["train"]["data"][:, :],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True
)
HTML(anim.to_jshtml())


In [ ]:
anim = plot_spatiotemporal_video(
    outputs["speckle_only"]["train"]["data"][:, ::2],
    batch_idx=3,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True
)
HTML(anim.to_jshtml())

In [ ]:
anim = plot_spatiotemporal_video(
    outputs["high_complexity"]["train"]["data"],
    batch_idx=2,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True
)
HTML(anim.to_jshtml())